In [1]:
from hana_ml import dataframe
cc = dataframe.ConnectionContext(userkey='MyDBKey')

In [2]:
import numpy as np
import pandas as pd
np.random.seed(2023)
data = pd.concat((pd.DataFrame(dict(ID=range(128),
                                    dates=pd.date_range(start='2022-02-02', periods=128))),
                  pd.DataFrame(np.random.normal(size=(128,2)), columns=['X1', 'X2'])),
                  axis=1)

In [3]:
from hana_ml.dataframe import create_dataframe_from_pandas
sim_df = create_dataframe_from_pandas(cc, data,
                                      "FFT_SIM_DATA_TBL",
                                      force=True)

100%|██████████| 1/1 [00:00<00:00,  2.71it/s]


In [4]:
from hana_ai.tools.hana_ml_tools.fft_tools import FFT
fft_tool = FFT(cc)

In [6]:
tool_input = dict(table_name='FFT_SIM_DATA_TBL',
                  key='dates',
                  real_col='X1',
                  imag_col='X2',
                  inverse=True)
fft_tool.run(tool_input=tool_input)

'{"fft_result_table": "FFT_SIM_DATA_TBL_FFT_RESULT"}'

In [7]:
cc.table("FFT_SIM_DATA_TBL_FFT_RESULT").collect()

,dates_int,REAL,IMAG
0,1,0.061002,-0.017000
1,2,0.025333,-0.025560
2,3,0.037085,-0.127404
3,4,-0.073787,-0.053339
4,5,-0.052561,0.091794
...,...,...,...
123,124,-0.011923,-0.090943
124,125,0.002024,-0.028287
125,126,-0.040987,0.062023
126,127,-0.121878,-0.032313


In [8]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [fft_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

C:\Users\I326292\AppData\Local\Temp\ipykernel_19448\1827641142.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)


In [10]:
instruction_str = "Please compute the inverse FFT of X1 and X2 in table FFT_SIM_DATA_TBL, " +\
"where key is dates, the number type X1 is real and the number type of X2 is imaginary."
agent_chain.invoke(instruction_str)

{'input': 'Please compute the inverse FFT of X1 and X2 in table FFT_SIM_DATA_TBL, where key is dates, the number type X1 is real and the number type of X2 is imaginary.',
 'output': 'The inverse FFT computation has been completed, and the results are stored in the table `FFT_SIM_DATA_TBL_FFT_RESULT`. You can check this table for the computed inverse FFT values. \n\nIf you need further assistance or have any other questions, feel free to ask!'}

In [11]:
cc.table('FFT_SIM_DATA_TBL_FFT_RESULT').collect()

,dates_int,REAL,IMAG
0,1,0.061002,-0.017000
1,2,0.025333,-0.025560
2,3,0.037085,-0.127404
3,4,-0.073787,-0.053339
4,5,-0.052561,0.091794
...,...,...,...
123,124,-0.011923,-0.090943
124,125,0.002024,-0.028287
125,126,-0.040987,0.062023
126,127,-0.121878,-0.032313


In [12]:
instruction_str = "Please compute the inverse FFT of X1 and X2 in table FFT_SIM_DATA_TABLE, " +\
"where key is dates, the number type X1 is real and the number type of X2 is imaginary."
agent_chain.invoke(instruction_str)

{'input': 'Please compute the inverse FFT of X1 and X2 in table FFT_SIM_DATA_TABLE, where key is dates, the number type X1 is real and the number type of X2 is imaginary.',
 'output': 'It seems that the table "FFT_SIM_DATA_TABLE" could not be found. Please ensure that the table name is correct and that it exists in the database. If you have any other questions or need further assistance, feel free to ask!'}

In [13]:
cc.drop_table("FFT_SIM_DATA_TBL_FFT_RESULT")
cc.drop_table("FFT_SIM_DATA_TBL")
cc.close()